In [1]:
###Load packages###
import pandas as pd
import os
import ast
from scipy import stats
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import numpy as np

###Load cleaned dataset###

#Set file paths
topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'Study1.0'
cleandir = os.path.join(topdir, f'data/{study}/Cleaned')
outputdir = os.path.join(topdir, f'Analysis/Plots/{study}')
outputdirCombined = os.path.join(topdir, f'data/Combined')

#Read in cleaned data (from cleaning scripts)
accommodate_path = os.path.join(cleandir, f'{study}Accommodate.csv')
predict_path   = os.path.join(cleandir, f'{study}Predict.csv')

df_accomodate = pd.read_csv(accommodate_path)
df_predict   = pd.read_csv(predict_path)

df_accomodate['task'] = 'accommodate'
df_predict['task']   = 'predict'


print("Accomodate columns:", df_accomodate.columns.tolist())
print("Predict columns:", df_predict.columns.tolist())


Accomodate columns: ['participant', 'free_texts', 'feedback', 'food_amount', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_tail', 'training_shape', 'training_color', 'testing_categories', 'conditionOrder', 'training_image_order', 'attention_check', 'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim', 'color_high', 'color_low', 'shape_high', 'shape_low', 'tail_high', 'tail_low', 'shape_discrete_slider.response', 'shape_direction_slider.response', 'shape_continuous_slider.response', 'color_discrete_slider.response', 'color_direction_slider.response', 'color_continuous_slider.response', 'tail_discrete_slider.response', 'tail_direction_slider.response', 'tail_continuous_slider.response', 'task']
Predict columns: ['participant', 'training_responses', 'food_amount', 'error', 'feedback', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_tail', 'training_shape', 'training_color', 'testing_categorie

In [2]:
#Converting string representations of lists back to lists

def parse_list_column(x):
    """take column entries that are strings representing lists and convert them to actual lists"""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            return ast.literal_eval(x)
        else:
            return [x]
    return []
for col in ['training_tail', 'training_shape', 'training_color', 'training_image_order', 'training_categories', 'testing_categories']:
    df_accomodate[col] = df_accomodate[col].apply(parse_list_column)
    df_predict[col]   = df_predict[col].apply(parse_list_column)

df_accomodate['testing_responses'] = df_accomodate['testing_responses'].apply(ast.literal_eval)
df_accomodate['food_amount'] = df_accomodate['food_amount'].apply(ast.literal_eval)
df_accomodate['testing_image_order'] = df_accomodate['testing_image_order'].apply(ast.literal_eval)
df_predict['testing_responses'] = df_predict['testing_responses'].apply(ast.literal_eval)
df_predict['food_amount'] = df_predict['food_amount'].apply(ast.literal_eval)
df_predict['testing_image_order'] = df_predict['testing_image_order'].apply(ast.literal_eval)
#Combine the dataframes and create an arbitrary column for participant numbering (the yoked orders are already stored in 'conditionOrder')
df_combined = pd.concat([df_accomodate, df_predict], ignore_index=True)
df_combined['participant'] = range(1, len(df_combined) + 1)



In [9]:
print(len(df_combined), "participants in combined dataset")

300 participants in combined dataset


Analysis #1: Overfitting

In [10]:
import pandas as pd

#Loop through rows and determine model parameter score for each participant

participant_rows = []

for _, row in df_combined.iterrows():
    tail_yes  = 1 if row['tail_discrete_slider.response']  == 'Yes' else 0
    shape_yes = 1 if row['shape_discrete_slider.response'] == 'Yes' else 0
    color_yes = 1 if row['color_discrete_slider.response'] == 'Yes' else 0

    model_param_score = tail_yes + shape_yes + color_yes

    participant_rows.append({
        'participant': row['participant'],
        'task': row['task'],  # predict vs accommodate
        'model_param_score': model_param_score,
        'conditionOrder': row['conditionOrder'],
        'overfit': model_param_score == 3, #overfit if all 3 dimensions selected
        'irrelevant_dim': row['irrelevant_dim']
    })

df_params = pd.DataFrame(participant_rows)

df_params.to_csv(os.path.join(outputdirCombined, 'df_params_for_study_1.csv'), index=False)

#Compare overfit vs not by condition
contingency = pd.crosstab(
    df_params['task'],
    df_params['overfit']
)

print(contingency)
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")


overfit      False  True 
task                     
accommodate     78     72
predict         80     70
Chi-square = 0.013
df = 1
p-value = 0.9079


In [11]:
contingency_all = pd.crosstab(
    df_params['task'],
    df_params['model_param_score']
)

print(contingency_all)

chi2, p, dof, expected = chi2_contingency(contingency_all)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")

ax = contingency_all.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_xlabel("Task")
ax.set_ylabel("Count")
#ax.set_title("Model Parameter Scores by Task")

# Add numerical labels with the count
for container in ax.containers:
    ax.bar_label(
        container,
        label_type="center",
        fontsize=9
    )
plt.legend(title="Model parameter score", bbox_to_anchor=(1.05, 1))
plt.tight_layout()

plt.close()



model_param_score  0   1   2   3
task                            
accommodate        8  16  54  72
predict            8  16  56  70
Chi-square = 0.065
df = 3
p-value = 0.9957


In [12]:
#Create map from short codes to feature descriptions

shape_map = {
    'square': 's',
    'circular': 'c'
}

color_map = {
    'blue': 'b',
    'yellow': 'y'
}

tail_map = {
    'curly': 't',
    'straight': 'n'
}


feature_maps = {
    'shape': shape_map,
    'color': color_map,
    'tail': tail_map
}


Analysis #2: Predicting importance scores

In [13]:
#Compute feature importance scores

from doctest import debug


def compute_feature_importance_from_df(df):
    """
    Compute numeric feature importance scores(-7 to 7) for each participant,
    based on the saved slider_responses and the feature _high/_low mapping.
    This is computed based on whether a feature was really relevant (positive sign) or irrelevant (negative sign).
    0 = no response or feature was not thought to be relevant
    columns:
      - shape_discrete_slider.response, shape_direction_slider.response, shape_continuous_slider.response
      - color_discrete_slider.response, ...
      - tail_discrete_slider.response, ...
      - shape_high, shape_low, color_high, color_low, tail_high, tail_low
    """
    features = ['shape', 'color', 'tail']
    
    def compute_row_importance(row, feat):
        disc = row[f'{feat}_discrete_slider.response']
        dirc = row[f'{feat}_direction_slider.response']
        cont = row[f'{feat}_continuous_slider.response']

        #If they said a feature wasn't relevant, then importance is 0
        
        if disc == 'No' or pd.isna(disc):
            return 0.0
        
        # Make sure continuous slider value exists, if not, set it to 0
        cont_val = float(cont) if not pd.isna(cont) else 0.0

        # Get mapping from long to short feature name
        mapping = feature_maps.get(feat, {})

        # Normalize strings: strip whitespace, collapse multiple spaces, lower-case
        def normalize_str(s):
            """Strip leading/trailing whitespace and collapse internal multiple spaces."""

            if isinstance(s, str):
                return " ".join(s.split()).lower()
            return ""

        #Name of features need to be normalized for comparison using the mapping
        dirc_norm = normalize_str(dirc)

        #Get internal short code for selected feature direction
        internal_dirc = mapping.get(dirc_norm, None)
        
        #Normalize high and low feature values from the dataframe
        high_val = normalize_str(row[f'{feat}_high']).lower()
        low_val  = normalize_str(row[f'{feat}_low']).lower()
        
        # Debug print statement (make sure mappings look right)
        debug = False
        if debug:
            print('response:', repr(dirc_norm), 'internal:', repr(internal_dirc), 
                'high:', repr(high_val), 'low:', repr(low_val))
            

        #If they correctly selected the high feature, assign positive sign
        if internal_dirc == high_val:
            sign = 1
        #If they incorrectly selected the low feature, assign negative sign
        elif internal_dirc == low_val:
            if debug:
                print('in negative')
            sign = -1
        else:
            if debug:
                print('in empty')
            sign = 0
            cont_val = 0.0

        # Add sign to continuous value
        importance = cont_val * sign

        return importance

    
    # Compute for each feature
    for feat in features:
        df[f'{feat}_importance'] = df.apply(lambda row: compute_row_importance(row, feat), axis=1)
    
    return df

df_combined = compute_feature_importance_from_df(df_combined)
if debug:
    print(df_combined['tail_importance'])

0      3.0
1     -5.0
2      5.0
3      4.0
4      6.0
      ... 
295    0.0
296    4.0
297   -2.0
298    0.0
299   -3.0
Name: tail_importance, Length: 300, dtype: float64


In [14]:
import pandas as pd
"""Reshape to long format with 1 row per participant x feature dimension"""
# Keep only necessary columns
cols_to_keep = [
    'participant', 'task', 
    'shape_importance', 'color_importance', 'tail_importance',
    'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim'
]

df_long = df_combined[cols_to_keep].copy()

# Melt importance columns
df_long = df_long.melt(
    id_vars=['participant', 'task', 'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim'],
    value_vars=['shape_importance', 'color_importance', 'tail_importance'],
    var_name='feature_dimension',
    value_name='feature_importance'
)

# Simplify feature dimension names
df_long['feature_dimension'] = df_long['feature_dimension'].str.replace('_importance','')

def get_relevance(row):
    if row['feature_dimension'] in [row['relevant_dim_1'], row['relevant_dim_2']]:
        return 'relevant'
    else:
        return 'irrelevant'

df_long['feature_relevance'] = df_long.apply(get_relevance, axis=1)

print(df_long)

     participant         task relevant_dim_1 relevant_dim_2 irrelevant_dim  \
0              1  accommodate          color           tail          shape   
1              2  accommodate          shape           tail          color   
2              3  accommodate           tail          shape          color   
3              4  accommodate           tail          color          shape   
4              5  accommodate           tail          color          shape   
..           ...          ...            ...            ...            ...   
895          296      predict           tail          color          shape   
896          297      predict          color          shape           tail   
897          298      predict          color          shape           tail   
898          299      predict          shape          color           tail   
899          300      predict          shape          color           tail   

    feature_dimension  feature_importance feature_relevance  
0

In [15]:
df_long.to_csv(os.path.join(outputdirCombined, 'df_long_for_R-Study1.csv'), index=False)

In [16]:
#Analyse the absolute value of feature importance ratings (ignoring direction)
df_longAbs = df_long.copy()
df_longAbs["abs_feature_importance"] = df_longAbs["feature_importance"].abs()
avg_importance_task = (
    df_longAbs
        .groupby(['task', 'feature_relevance'])['abs_feature_importance']
        .mean()
        .reset_index()
)
print(avg_importance_task)
avg_importance_relevance = (
    df_longAbs
        .groupby(['feature_relevance'])['abs_feature_importance']
        .mean()
        .reset_index()
)

print(avg_importance_relevance)

avg_importance_feature = (
    df_longAbs
        .groupby(['feature_dimension'])['abs_feature_importance']
        .mean()
        .reset_index()
)
print(avg_importance_feature)

          task feature_relevance  abs_feature_importance
0  accommodate        irrelevant                2.740000
1  accommodate          relevant                3.723333
2      predict        irrelevant                2.833333
3      predict          relevant                3.440000
  feature_relevance  abs_feature_importance
0        irrelevant                2.786667
1          relevant                3.581667
  feature_dimension  abs_feature_importance
0             color                3.083333
1             shape                3.476667
2              tail                3.390000


In [17]:
df_irrel = df_longAbs[df_longAbs["feature_relevance"] == "irrelevant"].copy()

accom_vals = df_irrel[df_irrel["task"]=="accommodate"]["abs_feature_importance"]
predict_vals = df_irrel[df_irrel["task"]=="predict"]["abs_feature_importance"]
mean_accom = np.mean(accom_vals)
mean_predict = np.mean(predict_vals)

print(f"Mean (accommodate) = {mean_accom:.3f}")
print(f"Mean (predict) = {mean_predict:.3f}")
t, p = ttest_ind(predict_vals, accom_vals, equal_var=False)  # Welch's t-test
print(f"t = {t:.3f}, p = {p:.4f}")
summary_df = pd.DataFrame({
    "task": ["accommodate", "predict"],
    "mean_abs_feature_importance": [mean_accom, mean_predict],
    "t_stat": [t, t], 
    "p_value": [p, p]    
})

# Save CSV
summary_df.to_csv(os.path.join(outputdir, "ttest_abs_featureimportance.csv"))

Mean (accommodate) = 2.740
Mean (predict) = 2.833
t = 0.350, p = 0.7263


Analyis #3: Predicting Error/Scores for items

In [3]:
#Group by average food amount per item in training
df = df_combined[['task', 'training_image_order', 'food_amount', 'conditionOrder']]
df_long2 = (
    df
    .explode(['training_image_order', 'food_amount'])
    .rename(columns={'training_image_order': 'item'})
)
avg_food = (
    df_long2
    .groupby(['task', 'conditionOrder', 'item'], as_index=False)
    ['food_amount']
    .mean()
)
avg_food_train = avg_food.copy()


In [4]:
#Get food consumption ratings by item

df = df_combined[['task', 'conditionOrder', 'testing_image_order', 'testing_responses', 'relevant_dim_1',
                  'relevant_dim_2', 'irrelevant_dim', 'color_high', 'tail_high', 'shape_high']]
df_long2 = (
    df
    .explode(['testing_image_order', 'testing_responses'])
    .rename(columns={'testing_image_order': 'item'})
)
avg_food_test = df_long2.copy()


In [5]:
#Now merge the two (actual food amounts in training vs ratings in testing) and compute error
df_merged = avg_food_test.merge(
    avg_food_train,
    on=['task', 'conditionOrder', 'item'],
    how='left'
)

#Add Error and absolute error
df_merged['error'] = (
    df_merged['testing_responses'] - df_merged['food_amount']
)
df_merged['abs_error'] = df_merged['error'].abs()
df_merged[['tail', 'color', 'shape']] = df_merged['item'].str.split('_', expand=True)

print(df_merged)

             task  conditionOrder   item testing_responses relevant_dim_1  \
0     accommodate              85  N_Y_S               5.0          color   
1     accommodate              85  N_Y_C               3.0          color   
2     accommodate              85  N_B_S               6.0          color   
3     accommodate              85  N_B_C               8.0          color   
4     accommodate              85  T_B_C               6.0          color   
...           ...             ...    ...               ...            ...   
2395      predict              13  N_Y_S               5.0          shape   
2396      predict              13  T_Y_S               7.0          shape   
2397      predict              13  T_Y_C               6.0          shape   
2398      predict              13  T_B_C               5.0          shape   
2399      predict              13  T_B_S               7.0          shape   

     relevant_dim_2 irrelevant_dim color_high tail_high shape_high  \
0    

In [21]:
df_merged.to_csv(os.path.join(outputdirCombined, 'df_merged_for_R_Study1.csv'), index=False)

In [6]:
# Columns indicating whether the item's feature is the "high" dimension (1 or 0 coding)
df_merged['tail_match_high']  = (df_merged['tail']  == df_merged['tail_high']).astype(int)
df_merged['color_match_high'] = (df_merged['color'] == df_merged['color_high']).astype(int)
df_merged['shape_match_high'] = (df_merged['shape'] == df_merged['shape_high']).astype(int)
#print(df_merged.head(20))
# Group by participant
participant_corrs = []

for pid, g in df_merged.groupby(['task', 'conditionOrder']):
    for feat in ['tail','color','shape']:
        # Column indicating match to high value
        match_col = f"{feat}_match_high"
        
        # Compute correlation
        corr = g['testing_responses'].corr(g[match_col])
        
        # Determine if this feature is relevant for this participant
        relevant = feat in [g['relevant_dim_1'].iloc[0], g['relevant_dim_2'].iloc[0]]
        
        # Store
        participant_corrs.append({
            'participant': pid,
            'task': g['task'].iloc[0],
            'feature_dimension': feat,
            'feature_relevance': 'relevant' if relevant else 'irrelevant',
            'correlation': corr,
            'irrelevant_dim': g['irrelevant_dim'].iloc[0],
            'abs_correlation': abs(corr) if pd.notna(corr) else None
        })

df_corr = pd.DataFrame(participant_corrs)




/opt/miniconda3/envs/PredictProj/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/miniconda3/envs/PredictProj/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [7]:
#code to get actual correlations between food amount and feature match for each participant in training
actual_corrs = []

for pid, g in df_merged.groupby(['task', 'conditionOrder']):

    for feat in ['tail','color','shape']:

        match_col = f"{feat}_match_high"

        corr = g['food_amount'].corr(g[match_col])

        relevant = feat in [
            g['relevant_dim_1'].iloc[0],
            g['relevant_dim_2'].iloc[0]
        ]

        actual_corrs.append({
            'participant': pid,
            'task': g['task'].iloc[0],
            'feature_dimension': feat,
            'feature_relevance':
                'relevant' if relevant else 'irrelevant',
            'actual_correlation': corr
        })

actual_corrs = pd.DataFrame(actual_corrs)
df_corr = df_corr.merge(
    actual_corrs[
        ['participant',
         'feature_dimension',
         'actual_correlation']
    ],
    on=['participant','feature_dimension']
)
print(df_corr.head(20))

         participant         task feature_dimension feature_relevance  \
0   (accommodate, 1)  accommodate              tail          relevant   
1   (accommodate, 1)  accommodate             color          relevant   
2   (accommodate, 1)  accommodate             shape        irrelevant   
3   (accommodate, 2)  accommodate              tail        irrelevant   
4   (accommodate, 2)  accommodate             color          relevant   
5   (accommodate, 2)  accommodate             shape          relevant   
6   (accommodate, 3)  accommodate              tail          relevant   
7   (accommodate, 3)  accommodate             color        irrelevant   
8   (accommodate, 3)  accommodate             shape          relevant   
9   (accommodate, 4)  accommodate              tail        irrelevant   
10  (accommodate, 4)  accommodate             color          relevant   
11  (accommodate, 4)  accommodate             shape          relevant   
12  (accommodate, 5)  accommodate              tail

In [12]:
from scipy.stats import ttest_ind

# Keep only the irrelevant feature
df_irrelevant = df_corr[
    df_corr["feature_relevance"] == "irrelevant"
].copy()

#take abs
df_irrelevant["abs_correlation"] = df_irrelevant["correlation"].abs()

predict = df_irrelevant.loc[
    df_irrelevant["task"] == "predict",
    "abs_correlation"
].dropna()

accommodate = df_irrelevant.loc[
    df_irrelevant["task"] == "accommodate",
    "abs_correlation"
].dropna()

print("Predict:")
print(f"  N = {len(predict)}")
print(f"  M = {predict.mean():.3f}")
print(f"  SD = {predict.std():.3f}")

print("\nAccommodate:")
print(f"  N = {len(accommodate)}")
print(f"  M = {accommodate.mean():.3f}")
print(f"  SD = {accommodate.std():.3f}")

# Welch  t-test
t, p = ttest_ind(
    predict,
    accommodate,
    equal_var=False
)

print("\nT-test:")
print(f"t = {t:.3f}")
print(f"p = {p:.4f}")

Predict:
  N = 149
  M = 0.303
  SD = 0.232

Accommodate:
  N = 148
  M = 0.314
  SD = 0.243

T-test:
t = -0.409
p = 0.6828


In [24]:
df_corr.to_csv(os.path.join(outputdirCombined, 'df_corr_for_R_Study1.csv'), index=False)